## Setup

Let's start by importing the libraries we'll need and loading the Ames housing dataset.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer


# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

print("✓ Libraries imported successfully")

### Load the Ames Housing Data

We'll use the Ames housing dataset throughout this notebook. This dataset contains information about house sales in Ames, Iowa.

In [ ]:
# Load data - adjust path if running in Google Colab
try:
    # Try local path first
    ames = pd.read_csv('../data/ames_clean.csv')
except FileNotFoundError:
    # If in Colab, load from GitHub
    url = 'https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_clean.csv'
    ames = pd.read_csv(url)

print(f"Dataset shape: {ames.shape}")
print(f"\nFirst few rows:")
ames.head()

Let's quickly explore the dataset structure:

In [ ]:
# Dataset information
print("Dataset Info:")
print(f"Number of rows: {len(ames):,}")
print(f"Number of columns: {len(ames.columns)}")
print(f"\nColumn types:")
print(ames.dtypes.value_counts())
print(f"\nTarget variable (SalePrice) summary:")
print(ames['SalePrice'].describe())

---

## 1. Encoding Categorical Variables

Machine learning algorithms work with numbers, not categories. We need to convert categorical variables into numerical format.

### 1.1 Dummy/One-Hot Encoding

Creates separate binary columns for each category. Best for nominal variables (no inherent order).

In [ ]:
# Look at the BldgType (building type) variable
print("Building types in the dataset:")
print(ames['BldgType'].value_counts())
print(f"\nNumber of unique building types: {ames['BldgType'].nunique()}")

In [ ]:
# Create dummy variables for building type
bldg_dummies = pd.get_dummies(ames['BldgType'], prefix='BldgType')

print("Dummy encoded building types (first 5 rows):")
print(bldg_dummies.head())
print(f"\nNumber of columns created: {len(bldg_dummies.columns)}")

In [ ]:
# Avoid the dummy variable trap by dropping the first category
bldg_dummies_safe = pd.get_dummies(ames['BldgType'],
                                   prefix='BldgType',
                                   drop_first=True)

print(f"Original columns: {len(bldg_dummies.columns)}")
print(f"After dropping first: {len(bldg_dummies_safe.columns)}")
print(f"\nColumns kept: {list(bldg_dummies_safe.columns)}")

💡 **Key Insight:** For linear regression, use `drop_first=True` to avoid multicollinearity (the dummy variable trap).

### 1.2 Label Encoding

Assigns a unique integer to each category, creating just one column. More compact but can be misleading for linear models.

In [ ]:
# Check how many neighborhoods we have
print(f"Number of unique neighborhoods: {ames['Neighborhood'].nunique()}")
print(f"\nSample neighborhoods:")
print(ames['Neighborhood'].value_counts().head())

In [ ]:
# Apply label encoding
le = LabelEncoder()
ames_encoded = ames.copy()
ames_encoded['Neighborhood_Encoded'] = le.fit_transform(ames['Neighborhood'])

# Show the mapping for a few examples
print("Label encoding results (first 10 rows):")
print(ames_encoded[['Neighborhood', 'Neighborhood_Encoded']].head(10))

⚠️ **Warning:** Never use label encoding for non-ordinal categorical variables with linear models! The model will incorrectly treat the numeric codes as having magnitude.

### 1.3 Ordinal Encoding

For categorical variables with a natural order, create custom mappings that preserve the ordering.

In [ ]:
# Look at the exterior quality variable
print("Exterior quality categories:")
print(ames['ExterQual'].value_counts().sort_index())

In [ ]:
# Create custom ordinal mapping that preserves the quality order
# Po = Poor, Fa = Fair, TA = Typical/Average, Gd = Good, Ex = Excellent
quality_map = {
    'Po': 1,  # Poor
    'Fa': 2,  # Fair
    'TA': 3,  # Typical/Average
    'Gd': 4,  # Good
    'Ex': 5   # Excellent
}

# Apply the mapping
ames_encoded['ExterQual_Encoded'] = ames['ExterQual'].map(quality_map)

# Show the results
print("Ordinal encoding results (first 10 rows):")
print(ames_encoded[['ExterQual', 'ExterQual_Encoded']].head(10))

In [ ]:
# Verify the encoding preserves order by checking mean prices
print("Mean sale price by exterior quality:")
quality_prices = ames_encoded.groupby('ExterQual_Encoded')['SalePrice'].mean().sort_index()
print(quality_prices)
print("\n✓ Higher quality ratings correspond to higher prices!")